# Cart-pole translation and swing-up excitation with Gaussian MPPI

The target places the cart at 20 m and the pole upright. A horizontal force accelerates the cart and excites the pendulum. Gaussian MPPI rolls out each force sequence under the nonlinear dynamics and favors sequences that reduce position and angle error. [MPPI-Generic example](https://github.com/ACDSLab/MPPI-Generic/blob/abcd1ab419cb27efa113573932adbde1e0380835/examples/cartpole_example.cu)

GTSAM Copyright 2010–2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415. All rights reserved.  
Authors: Frank Dellaert and Akash Ratheesh. See LICENSE for details.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam_unstable/examples/planning/CartPoleMPPI.ipynb)

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop plotly
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam
import plotly.graph_objects as go
from IPython.display import HTML, display

## Planning horizon and running costs

The cart and pole masses are 1 kg and the pole length is 1 m. The simulation starts at rest with the pole hanging downward. Angle is measured continuously from that downward equilibrium. The 100-step horizon and running costs match the upstream example. Its 5 N sampling spread and inverse temperature of 4 provide the starting settings for calibration. For fast CPU execution, this demonstration uses 64 candidates and 400 decisions (8 s of simulated motion), which shows cart translation and swing-up excitation; full upright stabilization typically takes 13–77 s (set `decisions = 1500`–`5000`, or `samples, decisions = 2048, 5000` for the upstream counts).

In [3]:
horizon, samples, decisions = 100, 64, 400
control_dim, dt = 1, 0.02
initial_state = np.zeros(4)  # position, velocity, pole angle, angular velocity
goal = np.array([20.0, 0.0, np.pi, 0.0])
coefficients = np.array([50.0, 10.0, 200.0, 1.0])
options = gtsam.GaussianProposalOptions()
options.includeNominal = True

Each state key carries a vector. A diagonal prior produces the desired running cost because GTSAM computes half the squared whitened residual. Scaling the noise sigmas by the square root of the horizon averages those costs over the horizon.

In [4]:
X = [gtsam.symbol("x", k) for k in range(horizon + 1)]
U = [gtsam.symbol("u", k) for k in range(horizon)]
graph = gtsam.NonlinearFactorGraph()
noise = gtsam.noiseModel.Diagonal.Sigmas(np.sqrt(horizon / (2 * coefficients)))
for key in X[:-1]:
    graph.add(gtsam.PriorFactorVector(key, goal, noise))
planner = gtsam.SamplingPlannerVectorVector(X, U, dt)

## Receding-horizon control

Each trial and each executed step clamps the applied force to the physical range of −5 to +5 N. The graph assigns a running penalty to deviations from the target at each step. Each decision executes the first control and shifts the remaining sequence forward.

In [5]:
def dynamics(state, control):
    _, speed, angle, rate = state
    force = float(np.clip(control[0], -5.0, 5.0))
    sine, cosine = np.sin(angle), np.cos(angle)
    denominator = 1.0 + sine**2
    acceleration = (force + sine * (rate**2 + 9.81 * cosine)) / denominator
    angular_acceleration = (-force * cosine - rate**2 * cosine * sine
                            - 2 * 9.81 * sine) / denominator
    return np.array([speed, acceleration, rate, angular_acceleration])

def plant_step(state, control):
    return state + dt * dynamics(state, control)

In [6]:
def simulate(sigma, beta, seed):
    x = initial_state.copy()
    proposal = gtsam.GaussianProposal(sigma**2 * np.eye(control_dim), samples, seed, options)
    rule = gtsam.MPPI(beta)
    history = [x.copy()]
    nominal = np.zeros((control_dim, horizon))
    for decision in range(decisions):
        initial = gtsam.Values()
        initial.insert(X[0], x)
        for k, key in enumerate(U):
            initial.insert(key, nominal[:, k].copy())
        plan = planner.optimize(graph, initial, dynamics, proposal, rule, 1)
        controls = np.clip(np.column_stack([plan.atVector(key) for key in U]), -5.0, 5.0)
        x = plant_step(x, controls[:, 0])
        history.append(x.copy())
        nominal = np.column_stack((controls[:, 1:], controls[:, -1]))
    history = np.asarray(history)
    assert np.isfinite(history).all()
    return history

def tracking_score(history):
    error = history[1:] - goal
    return np.mean((error**2) @ coefficients)

### Tuning the proposal

The plotted trial retains the upstream setting (`sigma = 5.0`, `beta = 4.0`) over the 400-step (8 s) window with the updated nominal controls clipped to `[-5, 5]` N before shifting so the plan does not saturate outside the actuator limits. Set `repeat_sweep = True` to run the nine-setting calibration sweep (standard deviations 2.5, 5.0, 7.5 × inverse temperatures 1, 4, 12 over seeds 100–103) and five-seed evaluation (seeds 42–46).

In [7]:
original = (5.0, 4.0)
selected = (2.5, 1.0)
repeat_sweep = False
if repeat_sweep:
    from itertools import product

    def median_score(setting, seeds):
        return np.median([tracking_score(simulate(*setting, seed)) for seed in seeds])

    grid = product((2.5, 5.0, 7.5), (1., 4., 12.))
    selected = min(grid, key=lambda setting: median_score(setting, range(100, 104)))
    for label, setting in [("Original", original), ("Selected", selected)]:
        print(label, setting, "evaluation score", median_score(setting, range(42, 47)))

history = simulate(*original, seed=42)
print("Final state", history[-1], "mean closed-loop cost", tracking_score(history))

Final state [27.2329422   0.34640255  0.37927174 -3.42620274] mean closed-loop cost 8855.689406795118


## Closed-loop trajectory

Position and angle measure separate goals. Reaching 20 m can precede sustained upright balance; the angular trace shows whether the pole stabilizes near π radians. The interactive figure uses a version-pinned Plotly CDN and requires an internet connection.

In [8]:
time = dt * np.arange(len(history))
fig = go.Figure()
fig.add_scatter(x=time, y=history[:, 0], name="Cart position (m)")
fig.add_scatter(x=time, y=history[:, 2], name="Pole angle (rad)")
fig.update_layout(xaxis_title="Time (s)", yaxis_title="Cart position (m) or pole angle (rad)")
from html import escape

document = fig.to_html(include_plotlyjs="cdn", config={"displaylogo": False, "responsive": True})
display(HTML(f'<iframe title="Swing-up and cart translation with Gaussian MPPI" style="width:100%;border:0" '
             f'height="{fig.layout.height or 480}" srcdoc="{escape(document, quote=True)}"></iframe>'))